# Geocoding

The following section breaks down the process of geocoding violation location address and respondent home address in the following steps:

**1. Building and Cleaning Addresses:** This section uses various fields from `violations_df_full` to build violation and respondent addresses appropriately, then clean them before geocoding

**2. Saving Intersections for Separate Geocoding:** This section creates dataframes `violation_intersections.csv` and `respondent_intersections.csv` for addresses that are intersections. These will require separate geocoding

**3. Running Addresses through NYC GeoSupport Geocoder:** This section uses async batching to send multiple requests, reducing runtime.

**4. Flagging Levels of Geocoding Precision:**
This section adds the following tier categorizations for the geocoded addresses:

`0` if no match
`1` for an exact match
`2` for a fallback match on the same street
`3` for a fallback match in the same neighborhood
`4` for a fallback match with the same ZIP code
`5` for a fallback match in the same borough
`nan` if the matched address is something else

In [ ]:
# Setup

import pandas as pd
import sys, os
sys.path.append(os.path.abspath("..")) 

DATA_DIR = "../data/processed"
os.makedirs(DATA_DIR, exist_ok=True)

#Load df and modify datetime
violations_df_full = pd.read_csv(f"{DATA_DIR}")
violations_df_full['violation_date'] = pd.to_datetime(violations_df_full['violation_date'], errors='coerce')
violations_df_full['violation_year'] = violations_df_full['violation_date'].dt.year

In [ ]:
#Only working with certain columns of the dataset

columns = ['ticket_number',
       'violation_date', 'violation_time', 'agency_normalized',
       'respondent_first_name', 'respondent_last_name',
       'balance_due', 'violation_location_borough',
       'violation_location_street_name', 'violation_location_state_name',
       'respondent_address_borough', 'respondent_address_house',
       'respondent_address_street_name', 'respondent_address_city',
       'respondent_address_zip_code', 'respondent_address_state_name',
       'hearing_result',
       'total_violation_amount', 'penalty_imposed', 'paid_amount',
       'additional_penalties_or_late_fees', 'compliance_status',
       'charge_1_code', 'charge_1_code_section', 'charge_1_code_description',
       'charge_1_infraction_amount', 'violation_location_house',
       'violation_location_city', 'violation_location_zip_code',
       'decision_location_borough', 'date_judgment_docketed',
       'violation_location_block_no', 'violation_location_lot_no',
       'violation_details','year']

final_df = violations_df_full[columns]

In [ ]:
#Deduplicating

print("Number of Duplicates:", final_df.duplicated().sum())
final_df = final_df.drop_duplicates()

## Building and Cleaning Address Strings